# Robotwin VV attention plotting pipeline

The plotting implementation lives in `render_vv_attention.py`; this notebook only configures and calls it. It follows the `26Mar26-PyramidForcing-frames72` visual style.

Matrix orientation:

- rows: concatenated current-video Q tokens, small to large from top to bottom;
- columns: chronological video-history K tokens, small to large from left to right;
- gray cells: future K fields that were empty in the CSV;
- one small figure per `(layer, head)`;
- one 4×6 head grid per layer.

The helper uses only NumPy and Matplotlib; it does not import PyTorch.

In [ ]:
from pathlib import Path
import sys


def find_project_root():
    cwd = Path.cwd().resolve()
    for candidate in (cwd, *cwd.parents):
        if all((candidate / name).is_dir() for name in ("notebooks", "data", "figures")):
            return candidate
    raise FileNotFoundError("Could not find jupyter-plot root")


PROJECT_ROOT = find_project_root()
WORKSET_NAME = "26Aug9-Robotwin-VV-attention"
HELPER_DIR = PROJECT_ROOT / "notebooks" / WORKSET_NAME
sys.path.insert(0, str(HELPER_DIR))

from render_vv_attention import (
    estimate_shared_vmax,
    load_head_matrix,
    locate_experiment,
    metadata,
    output_dir,
    render_experiment,
    render_head,
)

print(f"project : {PROJECT_ROOT}")
print(f"workset : {WORKSET_NAME}")

## Parameters

In [ ]:
ATTENTION_LINK = "attn-exp-8gpu"
EXPERIMENT_SLUG = None
FORMATS = ("png", "pdf")
COLOR_PERCENTILE = 99.5
COLOR_SAMPLE_FILES = 24
DPI = 300

# For a quick single-figure check, change these two values and run the next cell.
PREVIEW_LAYER = 0
PREVIEW_HEAD = 0

## Inspect the linked attention output

In [ ]:
EXPERIMENT_DIR, SUMMARY = locate_experiment(PROJECT_ROOT, ATTENTION_LINK, EXPERIMENT_SLUG)
EXPECTED_SHAPE, LAYERS, NUM_HEADS, ROW_BOUNDS, HISTORY_BOUNDS = metadata(EXPERIMENT_DIR, SUMMARY)
FIGURES_ROOT = PROJECT_ROOT / "figures" / WORKSET_NAME

print(f"experiment : {EXPERIMENT_DIR.name}")
print(f"shape      : {EXPECTED_SHAPE}")
print(f"chunks     : {SUMMARY['num_chunks']}")
print(f"layers     : {len(LAYERS)}")
print(f"heads      : {NUM_HEADS}")
print(f"Q bounds   : {ROW_BOUNDS}")
print(f"K bounds   : {HISTORY_BOUNDS}")

## Draw one figure

This cell draws exactly one small figure. Change `PREVIEW_LAYER` and `PREVIEW_HEAD` to inspect another head.

In [ ]:
V_MAX = estimate_shared_vmax(
    EXPERIMENT_DIR,
    EXPECTED_SHAPE,
    [PREVIEW_LAYER],
    [PREVIEW_HEAD],
    percentile=COLOR_PERCENTILE,
    max_files=1,
)
MATRIX = load_head_matrix(EXPERIMENT_DIR, EXPECTED_SHAPE, PREVIEW_LAYER, PREVIEW_HEAD)
SMALL_DIR = output_dir(FIGURES_ROOT, EXPERIMENT_DIR, PREVIEW_LAYER)
SMALL_PATHS = render_head(
    MATRIX,
    PREVIEW_LAYER,
    PREVIEW_HEAD,
    V_MAX,
    ROW_BOUNDS,
    HISTORY_BOUNDS,
    SMALL_DIR,
    formats=FORMATS,
    dpi=DPI,
)
print(*SMALL_PATHS, sep="\n")

## Draw all heads and one 4×6 figure per layer

This cell renders 30 × 24 individual head figures and 30 layer grids. For a 10-chunk output, create/update the corresponding data symlink and change `ATTENTION_LINK`.

In [ ]:
MANIFEST = render_experiment(
    PROJECT_ROOT,
    attention_link=ATTENTION_LINK,
    experiment_slug=EXPERIMENT_SLUG,
    layer_grid=True,
    formats=FORMATS,
    color_percentile=COLOR_PERCENTILE,
    color_sample_files=COLOR_SAMPLE_FILES,
    dpi=DPI,
)
MANIFEST